# Streamlit App Deployment Preprocessing

## Imports

In [31]:
import duckdb
import os
import numpy as np
import faiss
import pickle
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

## Initialize Paths

In [32]:
BASE_DIR    = ".."
SRC_DB      = os.path.join(BASE_DIR, "data/processed/amazon_reviews.duckdb")
DEPLOY_DIR  = os.path.join(BASE_DIR, "data/streamlitdeployment")
DEPLOY_DB   = os.path.join(DEPLOY_DIR, "amazon_reviews_deploy.duckdb")
FAISS_INDEX = os.path.join(DEPLOY_DIR, "faiss_index_deploy.bin")
FAISS_META  = os.path.join(DEPLOY_DIR, "faiss_meta_deploy.pkl")

N_PRODUCTS  = 15_000
BATCH_SIZE  = 1_000
MODEL_NAME  = "all-MiniLM-L6-v2"

os.makedirs(DEPLOY_DIR, exist_ok=True)
print("Deploy dir ready:", DEPLOY_DIR)

Deploy dir ready: ../data/streamlitdeployment


## Sample 20k Products from Source DB and Write to Deployment DB

In [33]:
for f in [DEPLOY_DB, FAISS_INDEX, FAISS_META]:         # delete tables if they exist
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted: {f}")

Deleted: ../data/streamlitdeployment/amazon_reviews_deploy.duckdb
Deleted: ../data/streamlitdeployment/faiss_index_deploy.bin
Deleted: ../data/streamlitdeployment/faiss_meta_deploy.pkl


In [34]:
src = duckdb.connect(SRC_DB, read_only=True)
                                               # randomly sample 20k products from meta
meta_sample = src.execute(f"""                   
    SELECT *
    FROM meta
    USING SAMPLE {N_PRODUCTS} ROWS
""").fetchdf()

print(f"Sampled {len(meta_sample):,} products")
meta_sample.head(3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sampled 15,000 products


,parent_asin,title,average_rating,rating_number,features,description,price,store,details,image_url
0,B00DF7IDFY,Sunset Rainbow Adult Urn,5.0,1,[],['7410-10 Features: -Made of aluminum. -Thread...,NaN,UrnsDirect2U,{'Product Dimensions': '6.5 x 6.5 x 11 inches'...,NaN
1,B013TEB5BG,Husqvarna Part Number 812000001 E-Ring,4.1,9,"['This is an O.E.M. authorized part', 'Fits va...",['This is an O.E.M. authorized part. Fits vari...,10.95,Husqvarna,"{'Product Dimensions': '1 x 1 x 1 inches', 'It...",NaN
2,B08C7LRSN3,Sun Energise 30W 12V Solar Panel Kits- 30 Watt...,3.8,34,['[Off Grid Kits]The 30w solar panel kits are ...,[],NaN,Sun Energise,"{'Brand': 'Sun Energise', 'Material': 'Monocry...",NaN


## Pull Only Top 5 Reviews per Product

In [35]:
asins = tuple(meta_sample['parent_asin'].tolist())

reviews_sample = src.execute(f"""
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY parent_asin 
                   ORDER BY helpful_vote DESC
               ) AS rn
        FROM reviews
        WHERE parent_asin IN {asins}
    ) ranked
    WHERE rn <= 5
""").fetchdf()

reviews_sample = reviews_sample.drop(columns=['rn'])
src.close()
print(f"Reviews for sampled products: {len(reviews_sample):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Reviews for sampled products: 43,243


## Write Both Tables to the Deployment DuckDB

In [36]:
deploy = duckdb.connect(DEPLOY_DB)

deploy.execute("CREATE OR REPLACE TABLE meta AS SELECT * FROM meta_sample")
deploy.execute("CREATE OR REPLACE TABLE reviews AS SELECT * FROM reviews_sample")

print("Tables written:")
print("  meta:   ", deploy.execute("SELECT COUNT(*) FROM meta").fetchone()[0])
print("  reviews:", deploy.execute("SELECT COUNT(*) FROM reviews").fetchone()[0])

Tables written:
  meta:    15000
  reviews: 43243


## Build meta_search with Merged Review Text

In [37]:
deploy.execute("""
    CREATE OR REPLACE TABLE meta_search AS
    SELECT 
        m.parent_asin,
        m.title,
        m.average_rating,
        m.price,
        m.store,
        m.image_url,
        (COALESCE(CAST(m.title AS VARCHAR), '') || ' ' ||
         COALESCE(CAST(m.features AS VARCHAR), '') || ' ' ||
         COALESCE(CAST(m.description AS VARCHAR), '') || ' ' ||
         COALESCE(r.review_text_agg, '')) AS page_content
    FROM meta m
    LEFT JOIN (
        SELECT 
            parent_asin,
            STRING_AGG(text, ' ') AS review_text_agg
        FROM (
            SELECT parent_asin, text,
                   ROW_NUMBER() OVER (
                       PARTITION BY parent_asin 
                       ORDER BY helpful_vote DESC
                   ) AS rn
            FROM reviews
            WHERE text IS NOT NULL
        ) ranked
        WHERE rn <= 5
        GROUP BY parent_asin
    ) r ON m.parent_asin = r.parent_asin
""")

deploy.execute("PRAGMA create_fts_index('meta_search', 'parent_asin', 'page_content', overwrite = 1)")
print("meta_search created:", deploy.execute("SELECT COUNT(*) FROM meta_search").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

meta_search created: 15000


## Build FAISS Index from meta_search

In [38]:
df = deploy.execute("""
    SELECT parent_asin, title, average_rating, price, store, image_url, page_content
    FROM meta_search
""").fetchdf()

texts    = df['page_content'].fillna('').tolist()
metadata = df[['parent_asin', 'title', 'average_rating', 'price', 'store', 'image_url']].to_dict(orient='records')

model = SentenceTransformer(MODEL_NAME)
all_embeddings = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Encoding"):
    batch      = texts[i:i + BATCH_SIZE]
    embeddings = model.encode(batch, show_progress_bar=False)
    all_embeddings.append(embeddings)

all_embeddings = np.vstack(all_embeddings).astype('float32')
index          = faiss.IndexFlatL2(all_embeddings.shape[1])
index.add(all_embeddings)

faiss.write_index(index, FAISS_INDEX)
with open(FAISS_META, 'wb') as f:
    pickle.dump(metadata, f)

print(f"FAISS index saved: {index.ntotal:,} vectors")

/home/ubu/miniforge3/envs/amz/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Encoding: 100%|██████████████████████████████████████████| 15/15 [36:36<00:00, 146.42s/it]

FAISS index saved: 15,000 vectors


## Check File Sizes

In [39]:
deploy.close()

for fname in [DEPLOY_DB, FAISS_INDEX, FAISS_META]:
    size_mb = os.path.getsize(fname) / (1024 * 1024)
    print(f"{os.path.basename(fname)}: {size_mb:.1f} MB")

amazon_reviews_deploy.duckdb: 82.8 MB
faiss_index_deploy.bin: 22.0 MB
faiss_meta_deploy.pkl: 2.6 MB
